# LangGraph Coordination Layer

Three-node `StateGraph` implementing Section 6.3:

```
get_classification_context  →  classify_ticket  →  resolve_ticket  →  END
```

Per question, writes 6 outputs into `answers_langgraph/<question_stem>/`:

- `classification_answer.txt` — raw text from node 1
- `resolution_answer.txt` — raw text from node 3
- `combined.json` — appendix log (ticket + both answers + structured classification)
- `classification_drift.json` — full DRIFT trace, classification call
- `resolution_drift.json` — full DRIFT trace, resolution call
- `coordination_log.json` — execution trace + criteria evidence (Section 7.3.3)

Run from the GraphRAG project root (the folder with `settings.yaml` and `output/`).


## Imports

In [14]:
import os
import json
import time
import asyncio
import inspect
import uuid
from pathlib import Path
from datetime import datetime
from typing import TypedDict, Literal, Any
try:
    from typing import NotRequired
except ImportError:
    from typing_extensions import NotRequired  # Python 3.10 fallback

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from graphrag.config.load_config import load_config
from graphrag.config.embeddings import (
    community_full_content_embedding,
    entity_description_embedding,
)
from graphrag.utils.api import get_embedding_store, load_search_prompt
from graphrag.query.indexer_adapters import (
    read_indexer_entities,
    read_indexer_reports,
    read_indexer_report_embeddings,
    read_indexer_text_units,
    read_indexer_relationships,
)
from graphrag.query.factory import get_drift_search_engine

load_dotenv()
print("Imports OK")


Imports OK


## Configuration

`ROOT` is resolved from `Path.cwd()`. The cwd must be your GraphRAG project root (with `settings.yaml` and `output/`).

In [15]:
ROOT          = Path.cwd()
QUESTIONS_DIR = ROOT / "QUERY_CAT1"
ANSWERS_DIR   = ROOT / "answers_langgraph"
COMMUNITY_LEVEL = 2

EXPECTED_NODE_SEQUENCE = [
    "get_classification_context",
    "classify_ticket",
    "resolve_ticket",
]

llm = ChatOpenAI(
    api_key=os.environ["GRAPHRAG_API_KEY"],
    model="gpt-5-mini",
    temperature=0,
)

print(f"ROOT          = {ROOT}")
print(f"QUESTIONS_DIR = {QUESTIONS_DIR}")
print(f"ANSWERS_DIR   = {ANSWERS_DIR}")


ROOT          = /Users/kompzen/Offline/uploadMTRepo/00_together/LangGraph_GraphRAG_Query
QUESTIONS_DIR = /Users/kompzen/Offline/uploadMTRepo/00_together/LangGraph_GraphRAG_Query/QUERY_CAT1
ANSWERS_DIR   = /Users/kompzen/Offline/uploadMTRepo/00_together/LangGraph_GraphRAG_Query/answers_langgraph


## DRIFT engine

Built once on first DRIFT call and reused across all questions in this session. Re-run this cell if you change the GraphRAG config or rebuild the index.

In [16]:
def _to_plain_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    return obj

def _load_prompt_compat(root, prompt_cfg):
    if len(inspect.signature(load_search_prompt).parameters) == 2:
        return load_search_prompt(root, prompt_cfg)
    return load_search_prompt(prompt_cfg)

def build_drift_engine(root: Path = ROOT, community_level: int = COMMUNITY_LEVEL):
    config = load_config(root)
    out_dir = root / "output"

    required = ["entities.parquet", "communities.parquet",
                "community_reports.parquet", "text_units.parquet",
                "relationships.parquet"]
    missing = [f for f in required if not (out_dir / f).exists()]
    if missing:
        raise FileNotFoundError(f"Missing in {out_dir.resolve()}: {missing}")

    entities_df          = pd.read_parquet(out_dir / "entities.parquet")
    communities_df       = pd.read_parquet(out_dir / "communities.parquet")
    community_reports_df = pd.read_parquet(out_dir / "community_reports.parquet")
    text_units_df        = pd.read_parquet(out_dir / "text_units.parquet")
    relationships_df     = pd.read_parquet(out_dir / "relationships.parquet")

    vs = config.vector_store
    if isinstance(vs, dict):
        vs_args = {name: _to_plain_dict(store) for name, store in vs.items()}
    else:
        vs_args = {"default": _to_plain_dict(vs)}
    description_store  = get_embedding_store(vs_args, entity_description_embedding)
    full_content_store = get_embedding_store(vs_args, community_full_content_embedding)

    entities      = read_indexer_entities(entities_df, communities_df, community_level)
    reports       = read_indexer_reports(community_reports_df, communities_df, community_level)
    read_indexer_report_embeddings(reports, full_content_store)
    text_units    = read_indexer_text_units(text_units_df)
    relationships = read_indexer_relationships(relationships_df)

    local_prompt  = _load_prompt_compat(root, config.drift_search.prompt)
    reduce_prompt = _load_prompt_compat(root, config.drift_search.reduce_prompt)

    return get_drift_search_engine(
        config=config,
        reports=reports,
        text_units=text_units,
        entities=entities,
        relationships=relationships,
        description_embedding_store=description_store,
        response_type="Multiple Paragraphs",
        local_system_prompt=local_prompt,
        reduce_system_prompt=reduce_prompt,
    )

_search = None

def _get_search():
    global _search
    if _search is None:
        print(f"[init] Building DRIFT engine from {ROOT}")
        _search = build_drift_engine()
        print("[init] DRIFT engine ready")
    return _search

print("DRIFT engine builder ready (will build lazily on first query)")


DRIFT engine builder ready (will build lazily on first query)


## DRIFT helper

Runs `search.search` twice per call — `reduce=False` for the traversal, `reduce=True` for the final answer. Returns `{question, final_answer, traversal, context_data}` matching the standalone drift script's format.

In [17]:
def _to_serializable(x):
    """Recursively convert DataFrames/Series/Path to JSON-serializable types."""
    if isinstance(x, pd.DataFrame):
        return x.to_dict(orient="records")
    if isinstance(x, pd.Series):
        return x.to_list()
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {k: _to_serializable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_serializable(v) for v in x]
    return x

def _df_to_records(x):  # kept for backward compat
    if isinstance(x, pd.DataFrame):
        return x.to_dict(orient="records")
    return x

def _clean_context_data(context_data):
    return _to_serializable(context_data)

def _json_default(o):
    if isinstance(o, pd.DataFrame):
        return o.to_dict(orient="records")
    if isinstance(o, pd.Series):
        return o.to_list()
    if isinstance(o, Path):
        return str(o)
    return str(o)

async def graphrag_drift_with_trace(query: str) -> dict:
    """Run DRIFT twice (trace + final answer)."""
    search = _get_search()
    trace_result = await search.search(query, reduce=False)
    final_result = await search.search(query, reduce=True)
    raw = {
        "question": query,
        "final_answer": final_result.response,
        "traversal": trace_result.response,
        "context_data": getattr(trace_result, "context_data", None),
    }
    # Deep-serialise so the LangGraph checkpointer can store this in state
    # without msgpack failing on nested pandas objects.
    return _to_serializable(raw)


## Stage prompts

Constants combined with the ticket text at runtime. Edit and re-run this cell to change the prompts — the next workflow run will pick them up immediately.

In [18]:
CLASSIFICATION_PROMPT = """\
You are reviewing an IT support ticket. Based on similar past cases in the
indexed knowledge base, classify this ticket as either first_level_support
or second_level_support.

- first_level_support: routine issues that were resolved without engineer
  involvement in similar past cases (standard procedures, documented fixes).
- second_level_support: complex issues that required engineer involvement
  or escalation in similar past cases.

Recommend a classification, briefly explain the reasoning by referring to
similar past cases, and cite the source documents you relied on.

--- Ticket ---
{ticket}
"""

RESOLUTION_PROMPT = """\
You are resolving an IT support ticket. Using the indexed organisational
knowledge — known fixes, documented procedures, and prior resolutions —
provide a clear, step-by-step resolution for the ticket below.

If the indexed knowledge does not contain enough information to fully
resolve the ticket, say so explicitly. Cite the source documents for
each step.

--- Ticket ---
{ticket}
"""


## Classification structuring

Schema + chain used by node 2 to convert GraphRAG's recommendation into a typed value. The LLM here does NOT classify — it only structures the recommendation GraphRAG already produced.

In [19]:
class TicketClassification(BaseModel):
    classification: Literal["first_level_support", "second_level_support"] = Field(
        description="Classification recommended by the GraphRAG response."
    )
    reason: str = Field(
        description="Short explanation including source citations from the GraphRAG response."
    )

structuring_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You convert a GraphRAG response into a typed classification result. "
     "You do NOT classify the ticket yourself — GraphRAG has already done so. "
     "Your job is only to extract the recommended classification and a short "
     "cited reason from the response provided."),
    ("human",
     "Ticket:\n{ticket}\n\nGraphRAG response:\n{context}"),
])

structuring_chain = structuring_prompt | llm.with_structured_output(TicketClassification)


## Shared state + nodes

Edit any node and re-run this cell + the next one (compile graph) to pick up the changes.

In [20]:
class AgentState(TypedDict):
    ticket_description: str
    classification_context: NotRequired[str]
    classification_drift:   NotRequired[dict]
    classification_output:  NotRequired[TicketClassification]
    resolution_output:      NotRequired[str]
    resolution_drift:       NotRequired[dict]


async def get_classification_context(state: AgentState) -> dict:
    print("--- Node 1: get_classification_context (DRIFT x2) ---")
    query = CLASSIFICATION_PROMPT.format(ticket=state["ticket_description"])
    drift = await graphrag_drift_with_trace(query)
    return {
        "classification_context": drift["final_answer"],
        "classification_drift":   drift,
    }

async def classify_ticket(state: AgentState) -> dict:
    print("--- Node 2: classify_ticket (structuring LLM) ---")
    result = await structuring_chain.ainvoke({
        "ticket": state["ticket_description"],
        "context": state["classification_context"],
    })
    return {"classification_output": result}

async def resolve_ticket(state: AgentState) -> dict:
    """Classification result is preserved in state but NOT consumed here
    (Section 6.3.3, second design choice)."""
    print("--- Node 3: resolve_ticket (DRIFT x2, no structuring) ---")
    query = RESOLUTION_PROMPT.format(ticket=state["ticket_description"])
    drift = await graphrag_drift_with_trace(query)
    return {
        "resolution_output": drift["final_answer"],
        "resolution_drift":  drift,
    }


## Compile graph

In [21]:
graph = StateGraph(AgentState)
graph.add_node("get_classification_context", get_classification_context)
graph.add_node("classify_ticket",            classify_ticket)
graph.add_node("resolve_ticket",             resolve_ticket)

graph.set_entry_point("get_classification_context")
graph.add_edge("get_classification_context", "classify_ticket")
graph.add_edge("classify_ticket",            "resolve_ticket")
graph.add_edge("resolve_ticket",             END)

rag_agent = graph.compile(checkpointer=MemorySaver())
print("Graph compiled")


Graph compiled


## Coordination log builder

Reads LangGraph's checkpoint history after a run and produces structured evidence for the three coordination criteria from Section 7.3.3:

- **Memory** — were upstream outputs still in state when `resolve_ticket` ran?
- **Execution and actuation** — was GraphRAG invoked separately per stage with task-specific prompts?
- **Orchestration and workflow control** — did nodes execute in the expected order and terminate after `resolve_ticket`?


In [22]:
def _query_preview(q, limit=300):
    if not q:
        return None
    return q[:limit] + ("..." if len(q) > limit else "")


# Mapping from "which node just executed" to "which state keys appear AFTER that node"
NODE_OUTPUT_KEYS = {
    "get_classification_context": {"classification_context", "classification_drift"},
    "classify_ticket":            {"classification_output"},
    "resolve_ticket":             {"resolution_output", "resolution_drift"},
}


def _infer_node_from_diff(prev_keys: set, curr_keys: set) -> str | None:
    """Figure out which node executed by looking at which state keys appeared.

    More robust than reading metadata['writes'] — works across LangGraph versions.
    """
    new_keys = curr_keys - prev_keys
    if not new_keys:
        return None
    for node, expected in NODE_OUTPUT_KEYS.items():
        if expected.issubset(new_keys):
            return node
    return None


def build_coordination_log(agent, thread_config, question_file):
    snapshots = list(agent.get_state_history(thread_config))
    snapshots.reverse()  # chronological

    steps = []
    observed_sequence = []
    prev_keys: set = set()

    for snap in snapshots:
        meta = snap.metadata or {}
        values = snap.values or {}
        curr_keys = set(values.keys())

        node = _infer_node_from_diff(prev_keys, curr_keys)
        wrote_keys = sorted(curr_keys - prev_keys)

        if node and node not in observed_sequence:
            observed_sequence.append(node)

        graphrag_invoked = False
        prompt_kind = None
        query_preview = None

        if node == "get_classification_context":
            drift = values.get("classification_drift")
            if isinstance(drift, dict) and drift.get("question"):
                graphrag_invoked = True
                prompt_kind = "classification"
                query_preview = _query_preview(drift["question"])
        elif node == "resolve_ticket":
            drift = values.get("resolution_drift")
            if isinstance(drift, dict) and drift.get("question"):
                graphrag_invoked = True
                prompt_kind = "resolution"
                query_preview = _query_preview(drift["question"])

        next_nodes = list(snap.next) if snap.next else []

        steps.append({
            "step": meta.get("step"),
            "timestamp": getattr(snap, "created_at", None),
            "source": meta.get("source"),
            "node_just_executed": node,
            "wrote_keys": wrote_keys,
            "state_keys_present": sorted(curr_keys),
            "graphrag_invoked": graphrag_invoked,
            "graphrag_prompt_kind": prompt_kind,
            "graphrag_query_preview": query_preview,
            "next_nodes": next_nodes,
        })

        prev_keys = curr_keys

    # Memory: when resolve_ticket is about to run (i.e. it's the next node),
    # are the upstream outputs already in state?
    pre_resolve_step = next(
        (s for s in steps if "resolve_ticket" in s["next_nodes"]),
        None,
    )
    cls_ctx_present = bool(
        pre_resolve_step and "classification_context" in pre_resolve_step["state_keys_present"]
    )
    cls_out_present = bool(
        pre_resolve_step and "classification_output" in pre_resolve_step["state_keys_present"]
    )

    invocations = [
        {"node": s["node_just_executed"],
         "prompt_kind": s["graphrag_prompt_kind"],
         "query_preview": s["graphrag_query_preview"]}
        for s in steps if s["graphrag_invoked"]
    ]
    invocations_per_stage = {}
    for inv in invocations:
        invocations_per_stage[inv["prompt_kind"]] = invocations_per_stage.get(inv["prompt_kind"], 0) + 1
    task_specific = "classification" in invocations_per_stage and "resolution" in invocations_per_stage

    sequence_matches = observed_sequence == EXPECTED_NODE_SEQUENCE
    last = steps[-1] if steps else None
    terminated = bool(
        last
        and last["node_just_executed"] == "resolve_ticket"
        and not last["next_nodes"]
    )

    return {
        "question_file": question_file,
        "thread_id": thread_config["configurable"]["thread_id"],
        "expected_node_sequence": EXPECTED_NODE_SEQUENCE,
        "observed_node_sequence": observed_sequence,
        "steps": steps,
        "criteria_evidence": {
            "memory": {
                "classification_context_present_at_resolve_ticket": cls_ctx_present,
                "classification_output_present_at_resolve_ticket": cls_out_present,
            },
            "execution_and_actuation": {
                "graphrag_invocations": invocations,
                "invocations_per_stage": invocations_per_stage,
                "task_specific_prompts": task_specific,
            },
            "orchestration_and_workflow_control": {
                "node_sequence_matches_expected": sequence_matches,
                "terminated_after_resolve_ticket": terminated,
            },
        },
    }


## Artefact writers + `run_one`

`run_one` returns `(qdir, state, coord_log)` so you can inspect a single test run inline.

In [23]:
def _write_text(path, text):
    path.write_text(text, encoding="utf-8")

def _write_json(path, payload):
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=_json_default),
        encoding="utf-8",
    )

def _write_artefacts(qdir, ticket, ticket_file, state, coord_log):
    cls = state["classification_output"]

    _write_text(qdir / "classification_answer.txt", state["classification_context"])
    _write_text(qdir / "resolution_answer.txt",     state["resolution_output"])

    _write_json(qdir / "combined.json", {
        "question_file": ticket_file,
        "ticket": ticket,
        "classification_answer":     state["classification_context"],
        "classification_structured": cls.model_dump(),
        "resolution_answer":         state["resolution_output"],
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    })

    _write_json(qdir / "classification_drift.json", state["classification_drift"])
    _write_json(qdir / "resolution_drift.json",     state["resolution_drift"])
    _write_json(qdir / "coordination_log.json",     coord_log)

async def run_one(question_path: Path, answers_root: Path = ANSWERS_DIR):
    """Run a single question and write all 6 artefacts. Returns (qdir, state, coord_log)."""
    qdir = answers_root / question_path.stem
    qdir.mkdir(parents=True, exist_ok=True)

    ticket = question_path.read_text(encoding="utf-8").strip()
    thread = {"configurable": {"thread_id": str(uuid.uuid4())}}

    state = await rag_agent.ainvoke({"ticket_description": ticket}, thread)
    coord_log = build_coordination_log(rag_agent, thread, question_path.name)

    _write_artefacts(qdir, ticket, question_path.name, state, coord_log)
    return qdir, state, coord_log


## Test on a single query

Use this to verify the workflow end-to-end before kicking off the full batch. Edit `TEST_QUESTION` to point at whichever ticket you want to try.

In [24]:
TEST_QUESTION = QUESTIONS_DIR / "query3.txt"

qdir, state, coord_log = await run_one(TEST_QUESTION)

print(f"\n✓ artefacts written to: {qdir}")
print(f"\nClassification: {state['classification_output'].classification}")
print(f"Reason: {state['classification_output'].reason[:200]}...")

print(f"\n--- Resolution preview (first 500 chars) ---")
print(state['resolution_output'][:500] + "...")

print(f"\n--- Coordination criteria ---")
print(json.dumps(coord_log['criteria_evidence'], indent=2))


--- Node 1: get_classification_context (DRIFT x2) ---
[init] Building DRIFT engine from /Users/kompzen/Offline/uploadMTRepo/00_together/LangGraph_GraphRAG_Query
[init] DRIFT engine ready


/Users/kompzen/Offline/LLM-Graphrag/GraphRAG-Index2/.venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
  0%|          | 0/8 [00:00<?, ?it/s]        No answer found for query: Show Conditional Access evaluation results for the affected sign-ins.
No follow-up actions found for response: {}
 12%|█▎        | 1/8 [00:31<03:39, 31.36s/it]No answer found for query: Show the sensitivity label configuration for the 'Confidential' classification and which Azure AD group(s) it allows.
No follow-up actions found for response: {}
 25%|██▌       | 2/8 [00:31<01:19, 13.27s/it]No answer found for query: Provide Azure AD provisioning and sync error logs for the last 48 hours
No follow-up actions found for response: {}
 75%|███████▌  | 6/8 [00:45<00:08,  4.19s/it]No answer found for query: Provide Teams client logs for users repo

--- Node 2: classify_ticket (structuring LLM) ---
--- Node 3: resolve_ticket (DRIFT x2, no structuring) ---


 38%|███▊      | 3/8 [00:30<00:38,  7.67s/it]No answer found for query: Which of the affected users (or an example user UPN) can you share so we can check whether they are direct members of IT‑Projekte vs showing transitive membership via SP_Project_IT in your exports?
No follow-up actions found for response: {}
 50%|█████     | 4/8 [00:31<00:21,  5.30s/it]No answer found for query: Can you confirm whether you want us to re-enable external FTP access for a limited test window (and, if yes, who is the authorized approver from the customer side)?
No follow-up actions found for response: {}
No follow-up actions for action: Can you confirm whether you want us to re-enable external FTP access for a limited test window (and, if yes, who is the authorized approver from the customer side)?
No follow-up actions for action: Which of the affected users (or an example user UPN) can you share so we can check whether they are direct members of IT‑Projekte vs showing transitive membership via SP_Proj


✓ artefacts written to: /Users/kompzen/Offline/uploadMTRepo/00_together/LangGraph_GraphRAG_Query/answers_langgraph/query3

Classification: second_level_support
Reason: GraphRAG recommends second_level_support because the SharePoint folder rename + sensitivity label producing per-user/Incognito-vs-normal access matches prior M365/Identity engineer escalations (AAD gr...

--- Resolution preview (first 500 chars) ---
Below I separate (A) what the indexed support records explicitly document about incidents that match your symptoms, and (B) a concise, evidence‑based step‑by‑step resolution plan you can run now (with the exact artifacts 2nd‑level engineers will need). Where a step is drawn from the indexed records I mark it as such and cite the record IDs; where a step is a recommended action not verbatim in the records I label it “recommended”.

Short summary / classification (direct)
- Based on prior incident...

--- Coordination criteria ---
{
  "memory": {
    "classification_context_pr

## Batch runner

`run_batch` walks every `*.txt` in `QUERY_GLOBAL/` and runs the workflow on each one. Skips questions whose `combined.json` already exists. Failures don't write `combined.json`, so failed questions auto-retry on the next run.

`batch_status` reports what's done and what's pending without doing any work — handy for checking progress before/after a batch.

In [25]:
def _is_done(q_file: Path, answers_root: Path = ANSWERS_DIR) -> bool:
    """A question is 'done' if its combined.json marker exists and is non-empty."""
    marker = answers_root / q_file.stem / "combined.json"
    return marker.exists() and marker.stat().st_size > 0


def batch_status(questions_dir: Path = QUESTIONS_DIR,
                 answers_root:  Path = ANSWERS_DIR,
                 show_next: int = 10) -> dict:
    """Show progress without running anything."""
    questions = sorted(questions_dir.glob("*.txt"))
    if not questions:
        print(f"No questions in {questions_dir}")
        return {"done": [], "pending": []}

    done    = [q.name for q in questions if _is_done(q, answers_root)]
    pending = [q.name for q in questions if not _is_done(q, answers_root)]

    print(f"Total questions in {questions_dir.name}/: {len(questions)}")
    print(f"  ✓ Done:     {len(done)}")
    print(f"  ◯ Pending:  {len(pending)}")

    if pending:
        print(f"\nNext up:")
        for p in pending[:show_next]:
            print(f"  ◯ {p}")
        if len(pending) > show_next:
            print(f"  ... and {len(pending) - show_next} more")

    return {"done": done, "pending": pending}


async def run_batch(questions_dir: Path = QUESTIONS_DIR,
                    answers_root:  Path = ANSWERS_DIR,
                    overwrite: bool = False) -> dict:
    if not questions_dir.exists():
        raise FileNotFoundError(f"Questions folder not found: {questions_dir}")
    answers_root.mkdir(parents=True, exist_ok=True)

    question_files = sorted(questions_dir.glob("*.txt"))
    if not question_files:
        print(f"No .txt questions found in {questions_dir}")
        return {"processed": [], "skipped": [], "failed": []}

    if overwrite:
        to_run, skipped_upfront = question_files, []
    else:
        to_run         = [q for q in question_files if not _is_done(q, answers_root)]
        skipped_upfront = [q for q in question_files if _is_done(q, answers_root)]

    total     = len(question_files)
    n_to_run  = len(to_run)
    n_skipped = len(skipped_upfront)

    print(f"Found {total} question(s) in {questions_dir.name}/")
    print(f"  Already done: {n_skipped}")
    print(f"  To run:       {n_to_run}")

    if n_to_run == 0:
        print("\nNothing to do — all questions already have answers.")
        return {"processed": [], "skipped": [q.name for q in skipped_upfront], "failed": []}

    print(f"\nStarting at {datetime.now().strftime('%H:%M:%S')}")
    batch_t0 = time.time()
    processed, failed = [], []

    for i, q_file in enumerate(to_run, start=1):
        upcoming = to_run[i].name if i < n_to_run else "— (last one)"
        print(f"\n[{i}/{n_to_run}] running: {q_file.name}   (next: {upcoming})")

        t0 = time.time()
        try:
            written, _, _ = await run_one(q_file, answers_root)
            elapsed = time.time() - t0
            print(f"[{i}/{n_to_run}] ✓ done   {q_file.name} in {elapsed:.1f}s  →  {written.name}/")
            processed.append(q_file.name)
        except Exception as e:
            elapsed = time.time() - t0
            print(f"[{i}/{n_to_run}] ✗ failed {q_file.name} after {elapsed:.1f}s: {e}")
            failed.append((q_file.name, str(e)))

    total_elapsed = time.time() - batch_t0
    avg = total_elapsed / max(1, n_to_run)
    print(f"\nFinished at {datetime.now().strftime('%H:%M:%S')}  "
          f"(total {total_elapsed:.1f}s, avg {avg:.1f}s/question)")
    print(f"Summary: {len(processed)} processed, {n_skipped} skipped, {len(failed)} failed")
    return {"processed": processed,
            "skipped":   [q.name for q in skipped_upfront],
            "failed":    failed}


## Check progress

Run this any time to see what's done and what's still pending. No work is performed — useful before kicking off `run_batch`, or after a partial run, or while planning which questions still need to go through.

In [26]:
batch_status()


Total questions in QUERY_CAT1/: 6
  ✓ Done:     1
  ◯ Pending:  5

Next up:
  ◯ query1.txt
  ◯ query2.txt
  ◯ query4.txt
  ◯ query5.txt
  ◯ query6.txt


{'done': ['query3.txt'],
 'pending': ['query1.txt',
  'query2.txt',
  'query4.txt',
  'query5.txt',
  'query6.txt']}

## Run the full batch

Each question prints `[i/N]` so you can see position in the run. To re-run a single question, delete its subfolder under `answers_langgraph/` (or call `run_batch(overwrite=True)` for everything).

In [ ]:
results = await run_batch()
results


Found 6 question(s) in QUERY_CAT1/
  Already done: 1
  To run:       5

Starting at 05:20:21

[1/5] running: query1.txt   (next: query2.txt)
--- Node 1: get_classification_context (DRIFT x2) ---


 12%|█▎        | 1/8 [00:29<03:29, 29.87s/it]No answer found for query: Do you want me to request a current external connectivity test (FTP/SFTP) from an external vantage point and to which IP/FQDN should that be targeted?
No follow-up actions found for response: {}
 62%|██████▎   | 5/8 [00:39<00:13,  4.54s/it]No answer found for query: Do you want me to draft a short escalation note (including the prior ticket references and the remediation steps used previously) for your Identity/AD team to request group consolidation and a controlled full sync?
No follow-up actions found for response: {}
 88%|████████▊ | 7/8 [00:45<00:03,  3.44s/it]No answer found for query: Provide the external auditor's delivery logs (protocol, endpoint, and error/timing details).
No follow-up actions found for response: {}
No follow-up actions for action: Provide the external auditor's delivery logs (protocol, endpoint, and error/timing details).
No follow-up actions for action: Do you want me to draft a short es

--- Node 2: classify_ticket (structuring LLM) ---
--- Node 3: resolve_ticket (DRIFT x2, no structuring) ---


 62%|██████▎   | 5/8 [00:33<00:13,  4.53s/it]No answer found for query: Should I coordinate the test window with the team who removed the firewall policy (Employee 28 / Network Team) to avoid conflicts or to document the planned temporary change?
No follow-up actions found for response: {}
No follow-up actions for action: Should I coordinate the test window with the team who removed the firewall policy (Employee 28 / Network Team) to avoid conflicts or to document the planned temporary change?
 12%|█▎        | 1/8 [00:25<03:00, 25.83s/it]No answer found for query: Please provide the Azure AD UPNs (or usernames) of two affected users so I can identify their device objects and cross‑check device attributes in the tenant.
No follow-up actions found for response: {}
 38%|███▊      | 3/8 [00:32<00:44,  8.82s/it]No answer found for query: Do you want a reproducible test to run now (add a temporary TEST user to the cloud group, verify access, then remove) to confirm the current state?
No foll

[1/5] ✓ done   query1.txt in 480.4s  →  query1/

[2/5] running: query2.txt   (next: query4.txt)
--- Node 1: get_classification_context (DRIFT x2) ---


...

  0%|          | 0/8 [00:00<?, ?it/s]No answer found for query: Can you provide the firewall/NAT log entries for 10.10.10.10:21 and the client source IP(s) for the same timestamps as the WinSCP tests?
No follow-up actions found for response: {}
 12%|█▎        | 1/8 [00:47<05:33, 47.68s/it]No answer found for query: Do you want the sign‑in query to include application/client filters (e.g., Outlook desktop client appId or Exchange Online client) or to search all clients for the UPN?
No follow-up actions found for response: {}
 25%|██▌       | 2/8 [00:49<02:03, 20.61s/it]No answer found for query: Should I include related federated/partner tenants (external partners) or only the primary tenant?
No follow-up actions found for response: {}
 38%|███▊      | 3/8 [00:49<00:56, 11.32s/it]No answer found for query: Are the other symptoms (SMTP send failures, chat presence, large-scan delivery, Basel printer queue) happening for the same users/groups or are they on different user sets/devices? (T

--- Node 2: classify_ticket (structuring LLM) ---
--- Node 3: resolve_ticket (DRIFT x2, no structuring) ---


 25%|██▌       | 2/8 [00:40<01:48, 18.16s/it]No answer found for query: Have any of the four UPNs been recently modified (password/credential rotation, role change, or Conditional Access exclusion) — if so, please provide timestamps or change request IDs.
No follow-up actions found for response: {}
 50%|█████     | 4/8 [00:43<00:26,  6.70s/it]No answer found for query: Can you export the SharePoint permission entry for SP_Compliance_Main (or a screenshot) showing the principal type and any identifier fields that SharePoint displays, and share the timestamp of that export?
No follow-up actions found for response: {}
No answer found for query: Do you have Azure AD sign‑in log examples (timestamps) for affected users showing CA evaluation outcomes and token/claim contents (or correlation IDs) so we can compare token claims to the SharePoint label/group requirements?
No follow-up actions found for response: {}
No follow-up actions for action: Have any of the four UPNs been recently modifie

[3/5] ✓ done   query4.txt in 648.8s  →  query4/

[4/5] running: query5.txt   (next: query6.txt)
--- Node 1: get_classification_context (DRIFT x2) ---


  0%|          | 0/8 [00:00<?, ?it/s]ExponentialRetry: Request failed, retrying, retries=1, delay=2.0, max_retries=10, exception=litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Traceback (most recent call last):
  File "/Users/kompzen/Offline/LLM-Graphrag/GraphRAG-Index2/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 1364, in embedding
    headers, sync_embedding_response = self.make_sync_openai_embedding_request(
  File "/Users/kompzen/Offline/LLM-Graphrag/GraphRAG-Index2/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/logging_utils.py", line 344, in sync_wrapper
    result = func(*args, **kwargs)
  File "/Users/kompzen/Offline/LLM